# Wine Quality Prediction using Machine Learning

## Project Overview
This notebook builds an end-to-end wine quality classification pipeline using physicochemical attributes from the wine dataset.

### Learning goals
- Load and clean data with pandas
- Explore the dataset using EDA and visualizations
- Create features and scale data for training
- Train and compare Logistic Regression and Random Forest models
- Evaluate classifier performance with metrics and graphics
- Save visuals for documentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Dataset
Load the wine quality dataset and inspect its structure.

In [ ]:
# Load the dataset
file_path = "dataset/winequality.csv"
df = pd.read_csv(file_path)

# Preview the dataset
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Display summary statistics for the numeric features
df.describe().T

## 2. Data Cleaning and Preprocessing
Check for missing values, remove unnecessary columns, and transform the target variable for modeling.

In [ ]:
# Check missing values and duplicate rows
print("Missing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Drop non-informative identifier column if present
if "Id" in df.columns:
    df = df.drop(columns=["Id"])

# Convert wine quality to a binary label: good (>=6) or bad (<6)
df["quality_label"] = np.where(df["quality"] >= 6, "good", "bad")

df["quality_label_encoded"] = df["quality_label"].map({"bad": 0, "good": 1})

df["quality_label"].value_counts()

## 3. Exploratory Data Analysis (EDA)
Visualize the target distribution and examine the relationships between features and wine quality.

In [ ]:
# Countplot of the binary quality labels
plt.figure(figsize=(8, 5))
sns.countplot(x="quality_label", data=df, palette=["#d9534f", "#5cb85c"])
plt.title("Distribution of Wine Quality Labels")
plt.xlabel("Quality Label")
plt.ylabel("Count")
plt.savefig("images/quality_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Display quality distribution for the original numeric quality score
quality_counts = df["quality"].value_counts().sort_index()
quality_counts

## 4. Correlation Analysis
Analyze feature correlations with a heatmap to identify strong predictors for quality.

In [ ]:
# Compute correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="vlag", cbar_kws={"shrink": .8})
plt.title("Correlation Heatmap of Wine Quality Features")
plt.savefig("images/correlation_heatmap.png", bbox_inches="tight")
plt.show()

## 5. Feature Selection and Train-Test Split
Select the feature set and split data into training and testing subsets.

In [ ]:
# Select features for modeling
feature_columns = [col for col in df.columns if col not in ["quality", "quality_label", "quality_label_encoded"]]
X = df[feature_columns]
y = df["quality_label_encoded"]

# Stratified split for balanced label distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 6. Feature Scaling
Scale the feature data so that models converge faster and perform more consistently.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns, index=X_test.index)

X_train_scaled.head()

## 7. Model Training
Train both Logistic Regression and Random Forest Classifier models for comparison.

In [ ]:
logistic_model = LogisticRegression(random_state=42, solver="liblinear")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

logistic_model.fit(X_train_scaled, y_train)
rf_model.fit(X_train_scaled, y_train)

## 8. Model Evaluation
Compare performance of both models using accuracy, classification reports, and confusion matrices.

In [ ]:
# Make predictions
logistic_preds = logistic_model.predict(X_test_scaled)
rf_preds = rf_model.predict(X_test_scaled)

# Calculate accuracy
logistic_accuracy = accuracy_score(y_test, logistic_preds)
rf_accuracy = accuracy_score(y_test, rf_preds)

print(f"Logistic Regression accuracy: {logistic_accuracy:.4f}")
print(f"Random Forest accuracy: {rf_accuracy:.4f}")

print("\nLogistic Regression Classification Report:\n")
print(classification_report(y_test, logistic_preds, target_names=["bad", "good"]))

print("Random Forest Classification Report:\n")
print(classification_report(y_test, rf_preds, target_names=["bad", "good"]))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["bad", "good"], yticklabels=["bad", "good"])
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.savefig(filename, bbox_inches="tight")
    plt.show()

plot_confusion_matrix(y_test, logistic_preds, "Logistic Regression Confusion Matrix", "images/confusion_matrix_logistic.png")
plot_confusion_matrix(y_test, rf_preds, "Random Forest Confusion Matrix", "images/confusion_matrix_random_forest.png")

## 9. Accuracy Comparison Visualization
Visualize how both models compare on test accuracy.

In [ ]:
model_names = ["Logistic Regression", "Random Forest"]
accuracies = [logistic_accuracy, rf_accuracy]

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=model_names, y=accuracies, palette=["#5a9bd5", "#ed7d31"])
ax.set_ylim(0, 1)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
for i, value in enumerate(accuracies):
    ax.text(i, value + 0.01, f"{value:.3f}", ha="center")

plt.savefig("images/model_accuracy_comparison.png", bbox_inches="tight")
plt.show()

## 10. Feature Importance
Identify the most important features using the Random Forest model.

In [ ]:
feature_importances = pd.Series(rf_model.feature_importances_, index=feature_columns)
feature_importances = feature_importances.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette="viridis")
plt.title("Feature Importance from Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.savefig("images/feature_importance.png", bbox_inches="tight")
plt.show()

## 11. Final Prediction Examples
Show example predictions using the trained models on held-out test data.

In [ ]:
sample_examples = X_test_scaled.head(6).copy()
sample_examples["Actual Label"] = y_test.loc[sample_examples.index].map({0: "bad", 1: "good"})
sample_examples["Logistic Prediction"] = logistic_model.predict(sample_examples[feature_columns])

# Apply mapping to human-readable labels
prediction_map = {0: "bad", 1: "good"}
sample_examples["Logistic Prediction"] = sample_examples["Logistic Prediction"].map(prediction_map)
sample_examples["Random Forest Prediction"] = rf_model.predict(sample_examples[feature_columns]).map(prediction_map)

sample_examples.reset_index(drop=True, inplace=True)
sample_examples.head()

## 12. Conclusion
This project demonstrates a complete machine learning pipeline for predicting wine quality. The Random Forest model typically provides stronger performance for this dataset, while Logistic Regression remains a reliable baseline model for binary classification.

### Key takeaways
- Data preprocessing and scaling are essential for consistent performance.
- Correlation analysis helps identify the most relevant features.
- Model comparison and visualization make it easier to choose the best classifier.

### Future work
- Extend the model to multi-class wine quality prediction.
- Explore feature engineering with polynomial features or interaction terms.
- Apply cross-validation and hyperparameter tuning for more robust results.